# Week 3 recap — Sheet 02: The rows that quietly disappear

Sheet 01 was about rows that **multiply**. This one is about rows that **vanish**
— and vanishing is worse, because a total that is too small looks like a smaller
business rather than a bug.

Week 3 hit this from four directions:

| Day | What disappeared | How |
|---|---|---|
| day 1, ws14 | rows with a null group key | `groupby` drops them, silently |
| day 3, ws03 | 21,117 rows matching `PRIORITY = 'High'` | the values carried quote characters |
| day 4, ws01 | 316 of 2,400 enrollments | a null category, dropped by `groupby` |
| day 4, ws07 | 156 enrollments | an inner join |

Not one of them raised. Every one produced a shorter, plausible, wrong answer.

Same superstore landing zone as sheet 01, in `data/bronze/`.

**Question 10 is supposed to raise an error.**

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Week 3 recap, sheet 02 — Rows that disappear. Run this once.
import glob
import numpy as np
import pandas as pd

BRONZE = "data/bronze/"
orders = (pd.concat([pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))],
                    ignore_index=True)
            .drop_duplicates(subset="LineID"))
customers = pd.read_csv(BRONZE + "customers.csv")
products = pd.read_csv(BRONZE + "products.csv")

# The real data has no missing keys. A real CRM export does, so this recap
# introduces them deliberately -- 60 customers lose their segment, and 25
# order lines lose their product. Both are DETERMINISTIC (every 30th and
# every 320th row), so every number below is reproducible.
customers_gappy = customers.copy()
customers_gappy.loc[customers_gappy.index % 30 == 0, "CustomerSegment"] = np.nan

orders_gappy = orders.reset_index(drop=True).copy()
orders_gappy.loc[orders_gappy.index % 320 == 0, "ProductID"] = np.nan

print("orders          ", orders.shape)
print("customers_gappy ", customers_gappy.shape,
      "| null CustomerSegment:", int(customers_gappy.CustomerSegment.isna().sum()))
print("orders_gappy    ", orders_gappy.shape,
      "| null ProductID:", int(orders_gappy.ProductID.isna().sum()))

PART A — the aggregate that shortens

### Question 1

Join `orders` to `customers_gappy` with a left join, then count order lines per `CustomerSegment` with a plain `groupby`. Print the result, its total, and the row count that went in.
> **NOTE:** add the reported numbers up before reading on.

In [ ]:
############################
## Your Code Here
############################

### Question 2

Find them. Count the joined rows whose `CustomerSegment` is null, and confirm the arithmetic.

In [ ]:
############################
## Your Code Here
############################

### Question 3

Fix it two ways: `groupby(..., dropna=False)`, and `fillna("Unknown")` before grouping. Print both, and say which you would ship.
> **NOTE:** day 4 worksheet 01 made this exact choice, and day 4 worksheet 08 made it a loading rule.

In [ ]:
############################
## Your Code Here
############################

PART B — the join that filters

### Question 4

Join `orders_gappy` to `products` three ways — `inner`, `left`, and `left` with `indicator=True` — and print the row count of each plus the indicator breakdown.

In [ ]:
############################
## Your Code Here
############################

### Question 5

Put money on it. Compute `SUM(Sales)` over the inner join and over `orders_gappy` itself, and print the difference and the percentage.
> **NOTE:** a revenue total that is too *small* is the hardest kind of wrong to notice.

In [ ]:
############################
## Your Code Here
############################

### Question 6

Do it properly: left join, then route the unmatched rows to an `Unknown` product rather than losing them. Print the row count, the total, and how many landed on `Unknown`.
> **NOTE:** day 4 worksheet 08's `Unknown` dimension member, applied here.

In [ ]:
############################
## Your Code Here
############################

PART C — values that are not what they look like

### Question 7

Day 3 found values carrying quote characters, so `WHERE PRIORITY = 'High'` matched nothing. Reproduce the shape of that here: pad `ProductCategory` with a trailing space on every row, then filter for `== "Technology"` and count.
> **NOTE:** an empty result is not an error. It reads exactly like "there were none".

In [ ]:
############################
## Your Code Here
############################

### Question 8

Write the check that would have caught question 7 in seconds: for every low-cardinality text column in the joined table, print its distinct values with `repr()`.
> **NOTE:** cheap enough to run on arrival, every time. It is how you find quoting, padding and casing before a query does.

In [ ]:
############################
## Your Code Here
############################

PART D — reconcile against something outside

### Question 9

Build a small pipeline — join, filter to 2012, aggregate by category — and print a reconciliation table: row count and `SUM(Sales)` at each step, with the difference explained at every stage.

In [ ]:
############################
## Your Code Here
############################

### Question 10

Finally, assert that the report in question 1 accounts for every row it was given. **This is supposed to fail.** Read the number and say what a reader of that report would have concluded.

In [ ]:
############################
## Your Code Here
############################